In [49]:
from pydantic import BaseModel
from typing import Literal
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage
model=ChatGroq(model="openai/gpt-oss-120b",temperature=0.1)


In [58]:
class Catagory(BaseModel):
    sheet_name:str
    category:Literal['BS','CF','Notes','PL','PPE','SOCIE','NOU']

class FileCategorised(BaseModel):
    files:list[Catagory]


prompt1 = ChatPromptTemplate.from_template(
    template="""Categories the sheet as useful or to drop according to these 6 Standard Sheets\n 
Balance Sheet (BS),
Cash and Flow (CF),
Notes (Notes),
Profit and loss (PL),
PPe_Schedule (PPE),
Change in equity (SOCIE),
Not of use in financial tagging (NOU)

available sheets {available_sheets}

"""

)

agent = prompt1 | model.with_structured_output(FileCategorised,strict=True)

In [59]:
import pandas as pd

sheets = pd.read_excel(
    "./user_data/BS DGPL Mar-2026-V1.xls",
    sheet_name=None
)

In [60]:
sheets_name=sheets.keys()

In [61]:
res=agent.invoke({"available_sheets":sheets_name})
res

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '**Sheet‑to‑Standard‑Category Mapping**\n\n| Original Sheet Name | Matched Standard Category | Keep / Drop* |\n|---------------------|---------------------------|--------------|\n| IND AS BS           | **BS** (Balance Sheet)    | Keep |\n| IND AS P&L          | **PL** (Profit & Loss)    | Keep |\n| Cash Flow Ind AS    | **CF** (Cash & Flow)      | Keep |\n| Change in Equity    | **SOCIE** (Change in Equity) | Keep |\n| Policies            | **NOU** (Not useful)      | Drop |\n| FA                  | **NOU** (Not useful)      | Drop |\n| Notes of IND AS BS  | **Notes**                 | Keep |\n| Notes of IND AS P&L | **Notes**                 | Keep |\n| Notes to accounts   | **Notes**                 | Keep |\n| RPT                 | **NOU** (Not useful)      | Drop |\n| Working Cash        

In [62]:
df = pd.read_excel("./canonical_sheets/CANONICAL_BS.xlsx")

/tmp/ipykernel_10584/4075667835.py:1: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  df = pd.read_excel("./canonical_sheets/CANONICAL_BS.xlsx")


,Unnamed: 0,DIXON GLOBAL PRIVATE LIMITED,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31
0,NaN,CIN: U51900UP2010PTC042400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"BALANCE SHEET AS AT 31ST MARCH, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,(Rs. in Lakh unless otherwise stated),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Particulars,NaN,NaN,Note \nNo.,NaN,"As at 30Th SEPTEMBER,2019",NaN,As at \n31st March 2026,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
421,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
sheets['IND AS BS '].to_excel("indas.xlsx")

In [77]:
prompt = f"""
You are a structure analyser your task is to write apython pandas code which can transform

this file {sheets['IND AS BS '].info()}
{sheets['IND AS BS '].sample(4)}
whose file path is /home/vanshsharma/projects/XBRL_autopilot/notebooks/indas.xlsx
into this ones structure only but data should be of previous one ok

{df.info()}
{df.sample(4)}

write the code by passing in the tool
"""




<class 'pandas.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 32 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    0 non-null      float64
 1   DIXON GLOBAL PRIVATE LIMITED  40 non-null     str    
 2   Unnamed: 2                    12 non-null     str    
 3   Unnamed: 3                    1 non-null      str    
 4   Unnamed: 4                    24 non-null     object 
 5   Unnamed: 5                    14 non-null     str    
 6   Unnamed: 6                    27 non-null     object 
 7   Unnamed: 7                    0 non-null      float64
 8   Unnamed: 8                    28 non-null     object 
 9   Unnamed: 9                    5 non-null      str    
 10  Unnamed: 10                   24 non-null     object 
 11  Unnamed: 11                   28 non-null     object 
 12  Unnamed: 12                   1 non-null      str    
 13  Unnamed: 13     

In [78]:
from pathlib import Path
import subprocess
import sys
from langchain_core.tools import tool


@tool
def write_and_run_code(code: str, filename: str = "t.py", timeout: int = 15) -> str:
  """Write Python code to a file (default: t.py), execute it, and return stdout, stderr, or errors to the agent."""
  target_path = Path(filename)

  try:
    target_path.write_text(code, encoding="utf-8")
  except OSError as exc:
    return f"[File Write Error] Failed to write '{target_path}': {exc}"

  try:
    result = subprocess.run(
        [sys.executable, str(target_path)],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
  except subprocess.TimeoutExpired as exc:
    partial_out = (exc.stdout or "").strip()
    partial_err = (exc.stderr or "").strip()
    return (
        f"[Execution Timeout] Script exceeded {timeout}s limit.\n"
        f"Partial STDOUT:\n{partial_out or '(empty)'}\n"
        f"Partial STDERR:\n{partial_err or '(empty)'}"
    )
  except Exception as exc:
    return f"[Execution Error] Unexpected failure running '{target_path}': {exc}"

  stdout = result.stdout.strip()
  stderr = result.stderr.strip()

  response_parts = [
      f"File: {target_path}",
      f"Exit Code: {result.returncode}",
      f"STDOUT:\n{stdout if stdout else '(empty)'}",
  ]
  if stderr:
    response_parts.append(f"STDERR / Traceback:\n{stderr}")

  return "\n\n".join(response_parts)

In [79]:
prompt

'\nYou are a structure analyser your task is to write apython pandas code which can transform\n\nthis file None\n     Unnamed: 0 DIXON GLOBAL PRIVATE LIMITED Unnamed: 2 Unnamed: 3 Unnamed: 4  \\\n200         NaN                          NaN        NaN        NaN        NaN   \n288         NaN                          NaN        NaN        NaN        NaN   \n414         NaN                          NaN        NaN        NaN        NaN   \n322         NaN                          NaN        NaN        NaN        NaN   \n\n    Unnamed: 5 Unnamed: 6  Unnamed: 7 Unnamed: 8 Unnamed: 9  ... Unnamed: 22  \\\n200        NaN        NaN         NaN        NaN        NaN  ...         NaN   \n288        NaN        NaN         NaN        NaN        NaN  ...         NaN   \n414        NaN        NaN         NaN        NaN        NaN  ...         NaN   \n322        NaN        NaN         NaN        NaN        NaN  ...         NaN   \n\n    Unnamed: 23 Unnamed: 24 Unnamed: 25 Unnamed: 26 Unnamed: 27  U

In [83]:
from langgraph.prebuilt import create_react_agent


agent = create_react_agent(
    model=model,
    tools=[write_and_run_code],
    prompt=prompt,
)

/tmp/ipykernel_10584/4241988712.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [84]:
res=agent.invoke({"messages": [("user", "continue")]})
res

{'messages': [HumanMessage(content='continue', additional_kwargs={}, response_metadata={}, id='91ae4c3e-4147-47c3-be22-8d6e1d25881e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to write pandas code to transform the first file (indas.xlsx) which appears to have many unnamed columns and NaNs, into the structure of the second example: columns Row_Index, Line_Item, Breadcrumb_Hierarchy, Note_No, Current_Year_Amount, Previous_Year_Amount. We need to map data from the first file to these columns. However we don't know the actual data; the example shows all NaNs. Probably the first file is raw XBRL data with many columns, need to reshape.\n\nWe need to write code that reads the Excel file, maybe drop unnamed columns, rename columns, maybe melt.\n\nWe can assume the first file has columns like 'Row_Index', 'Line_Item', etc but they are currently unnamed. So we need to assign proper column names based on some logic: maybe the first column is index, second is line 

In [89]:
res['messages'][-1].content

'Below is a **stand‑alone Python script** that will read the raw `indas.xlsx` file, **detect the real header row**, reshape the data and finally write a clean file that has exactly the columns you showed in the target example:\n\n```python\n#!/usr/bin/env python3\n"""\nTransform  /home/vanshsharma/projects/XBRL_autopilot/notebooks/indas.xlsx\ninto a tidy table with the columns\n\n    Row_Index\n    Line_Item\n    Breadcrumb_Hierarchy\n    Note_No\n    Current_Year_Amount\n    Previous_Year_Amount\n\nThe script works even when the original workbook does not have a proper\nheader (as in the snippet you posted).  It automatically:\n\n1. Loads the sheet without assuming any header.\n2. Scans the rows to locate the first row that contains *any* of the\n   expected column names.\n3. Uses that row as the header, drops everything above it and any\n   completely‑empty rows/columns.\n4. Keeps only the six columns we need (renaming them to the exact\n   canonical names above, even if the source u

In [90]:
with open("t.py","w") as f:
    f.write(res['messages'][-1].content)